# COBOL Intelligence Platform — Fine-Tuning on Colab

Fine-tune CodeLlama-7B with QLoRA on COBOL explanation data generated by the cobol-intel pipeline.

**Requirements:** Colab with GPU runtime (T4 is sufficient).

**Estimated time:** ~30-50 minutes total.

## Step 0 — Set GPU Runtime

1. Go to **Runtime > Change runtime type**
2. Select **T4 GPU**
3. Click **Save**

Then run the cell below to verify:

In [ ]:
!nvidia-smi

## Step 1 — Install Dependencies

In [ ]:
!pip install -q \
    torch \
    transformers>=4.40.0 \
    peft>=0.11.0 \
    datasets>=2.19.0 \
    accelerate>=0.30.0 \
    bitsandbytes>=0.43.0 \
    huggingface_hub

## Step 2 — Clone Repo & Install cobol-intel

In [ ]:
!git clone https://github.com/WwzFwz/finoss-cobol-intel.git
%cd finoss-cobol-intel
!pip install -q -e .

## Step 3 — Generate Training Dataset

Runs all 14 COBOL samples through the analysis pipeline, then generates instruction-tuning pairs.

In [ ]:
!python tools/dataset_builder.py \
    --samples-dir samples \
    --copybook-dir copybooks \
    --output dataset \
    --format both

In [ ]:
# Verify dataset
!wc -l dataset/cobol_instruct_alpaca.jsonl
!head -1 dataset/cobol_instruct_alpaca.jsonl | python -m json.tool | head -20

## Step 4 — Fine-Tune with QLoRA

4-bit quantized CodeLlama-7B + LoRA adapters. Fits comfortably on T4 (16GB VRAM).

**This takes ~15-30 minutes.**

In [ ]:
!python tools/finetune.py \
    --dataset dataset/cobol_instruct_alpaca.jsonl \
    --base-model codellama/CodeLlama-7b-Instruct-hf \
    --output models/cobol-explain-7b \
    --epochs 3 \
    --batch-size 2 \
    --gradient-accumulation 8 \
    --lr 2e-4 \
    --lora-r 16 \
    --lora-alpha 32 \
    --max-length 2048 \
    --quantize \
    --eval-split 0.1

## Step 5 — Quick Sanity Test

Load the fine-tuned model and test with a sample prompt.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

base_model = "codellama/CodeLlama-7b-Instruct-hf"
adapter_path = "models/cobol-explain-7b"

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    ),
    device_map="auto",
)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

prompt = """### Instruction:
Given the following structured analysis artifacts from COBOL program PAYMENT, provide a business-level explanation for stakeholders.

### Input:
Program PAYMENT contains 5 paragraphs and 12 data items.
Business rules: IF WS-BALANCE > 0 PERFORM PROCESS-PAYMENT.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Step 6 — Push to HuggingFace

1. Create a HuggingFace account at https://huggingface.co/join
2. Create an access token at https://huggingface.co/settings/tokens (Write permission)
3. Run the cell below and paste your token when prompted

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Change this to your HuggingFace username
HF_USERNAME = "WwzFwz"
REPO_NAME = f"{HF_USERNAME}/cobol-explain-7b"

from huggingface_hub import HfApi

api = HfApi()
api.create_repo(REPO_NAME, exist_ok=True)
api.upload_folder(
    folder_path="models/cobol-explain-7b",
    repo_id=REPO_NAME,
    commit_message="Upload cobol-explain-7b LoRA adapter (QLoRA, CodeLlama-7B)",
)
print(f"\nModel uploaded to: https://huggingface.co/{REPO_NAME}")

## Done!

Your fine-tuned model is now on HuggingFace. Users can use it with:

```bash
pip install "cobol-intel[local]"
cobol-intel explain samples/complex/payment.cbl --model local
```